# 7) Reservoirs and pumped hydro — Exercises

**Goals:** simulate a reservoir's level and the pumping break-even rule by hand, build your own reservoir model and match PyPSA, see how the water value responds to wet and dry weeks and to a small reservoir, and size a pumped-storage plant.

The setup cell holds the lecture's week (`LOAD`, `FREE`, `THERMAL`, a 1,000 MW turbine) and a `pypsa_week` helper for checking. Power in MW, energy in MWh, prices and water values in €/MWh, cost in € per week.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- the lecture's week (same seed, same numbers) ----
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
LOAD = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
FREE = (2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
        + 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9) + 300.0)
THERMAL = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                       index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))
TURBINE = 1000.0          # MW


def pypsa_week(inflow=120.0, e_nom=30_000.0, pumped=None):
    """The lecture's week in PyPSA, for checking. Returns the solved network."""
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, r in THERMAL.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=TURBINE, max_hours=e_nom / TURBINE, p_min_pu=0.0,
          inflow=inflow, cyclic_state_of_charge=True)
    if pumped:
        n.add("StorageUnit", "pumped", bus="HR", p_nom=pumped["p_nom"], max_hours=pumped["e_nom"] / pumped["p_nom"],
              efficiency_store=pumped["eta"], efficiency_dispatch=pumped["eta"], cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n

### Warm-ups

**1. Following the water**

Write `simulate_level(start, inflow, turbine, spill=None)` returning the level after each hour, e[t] = e[t−1] + inflow[t] − turbine[t] − spill[t]. Then check that the lecture's statement holds: a turbine that produces exactly the week's inflow returns the reservoir to where it started.

In [ ]:
def simulate_level(start: float, inflow, turbine, spill=None) -> np.ndarray:
    """Reservoir level (MWh) at the end of each hour."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.allclose(simulate_level(100, 50, [0, 0, 300]), [150, 200, -50]), "a level below zero means the plan is impossible"
plan = np.zeros(T)
plan[23::24] = 120.0 * 24                                       # each day's whole inflow in its last hour
levels = simulate_level(0.0, 120.0, plan)
assert np.isclose(levels[-1], 0.0) and levels.min() >= -1e-9, "inflow in = water out: back to the start"
assert np.isclose(levels.max(), 120.0 * 23), "the level peaks just before each release"
early = np.zeros(T)
early[19] = 120.0 * 24                                          # a full day's water at 19:00 on day 1 ...
assert simulate_level(0.0, 120.0, early).min() < 0, "... is more than has arrived by then: an impossible plan"

**2. When does pumping pay?**

Write `pumping_pays(price_pump, price_generate, round_trip)` (True if one more round trip lowers cost) and `break_even_pump_price(price_generate, round_trip)`. Check the lecture's 66.8 €/MWh.

In [ ]:
def pumping_pays(price_pump: float, price_generate: float, round_trip: float) -> bool:
    """True if pumping 1 MWh at price_pump and selling round_trip MWh at price_generate gains money."""
    ...  # YOUR CODE HERE


def break_even_pump_price(price_generate: float, round_trip: float) -> float:
    """Highest price at which pumping still pays (EUR/MWh)."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(break_even_pump_price(89.1, 0.866 ** 2), 66.8, atol=0.05), "the lecture's 66.8 EUR/MWh"
assert pumping_pays(12.0, 89.1, 0.75) and not pumping_pays(70.0, 89.1, 0.75)
assert not pumping_pays(0.0, 0.0, 0.75), "no spread, no profit"

### Core

**3. Your own reservoir, and wet and dry weeks**

Write `reservoir_week(inflow, e_nom)` — the lecture's Linopy model without pumped hydro. Return a dict with the cost, the water value per hour (positive, €/MWh), the turbine output and the level. Check it against PyPSA, then sweep the average inflow from a very dry to a very wet week. How does the water value respond?

In [ ]:
def reservoir_week(inflow: float = 120.0, e_nom: float = 30_000.0) -> dict:
    """Cost (EUR), water value (EUR/MWh per hour), turbine (MW) and level (MWh) of the week."""
    ...  # YOUR CODE HERE

In [ ]:
for inflow in [120.0, 480.0]:
    assert np.isclose(reservoir_week(inflow)["cost"], pypsa_week(inflow).objective, rtol=1e-9), inflow
sweep = pd.DataFrame({inflow: {"cost": (r := reservoir_week(inflow))["cost"], "water_value": r["water_value"].mean(),
                               "hours_running": int((r["turbine"] > 1e-6).sum())}
                      for inflow in [30.0, 60.0, 120.0, 240.0, 480.0, 960.0]}).T.rename_axis("inflow_MW")
assert sweep.water_value.is_monotonic_decreasing, "more water is worth less per MWh"
assert sweep.cost.is_monotonic_decreasing
sweep.round(1)

**4. A small reservoir**

Now keep the lecture's inflow (120 MW) but shrink the reservoir. Find the largest size in [400, 800, 1,600, 3,200, 6,400] MWh at which the water value is **no longer the same in every hour**, and show how it varies over the week. (Hint: compare with the highest level the lecture's optimal schedule reached.)

In [ ]:
def varying_water_value(sizes) -> pd.DataFrame:
    """Number of distinct water values and their range for each reservoir size."""
    ...  # YOUR CODE HERE

In [ ]:
table = varying_water_value([400, 800, 1600, 3200, 6400])
largest_varying = table.index[table.distinct_values > 1].max()
assert table.loc[6400, "distinct_values"] == 1 and table.loc[3200, "distinct_values"] > 1, "the threshold lies between"
assert np.isclose(reservoir_week(120.0, 1e9)["cost"], reservoir_week(120.0, 30_000.0)["cost"]), "a huge reservoir adds nothing here"
small = reservoir_week(120.0, float(largest_varying))
print(f"the water value varies once the reservoir is {largest_varying:.0f} MWh or smaller")
print("water value by day (mean, EUR/MWh):", np.round(small["water_value"].reshape(7, 24).mean(axis=1), 1))
table.round(1)

**5. Interpretation**

Explain the steps in the inflow sweep: why does the water value fall in jumps rather than smoothly, and what do the jump levels correspond to? And why does a small reservoir give different water values on different days, while the lecture's large one gives a single value?

*Your answer here.*

### Challenge

**6. How large should the upper basin be?**

Add a 300 MW pumped-storage plant (0.866 efficiency each way) to the lecture's week. Write `pumped_value(sizes)` returning, for each upper-basin size (MWh), the weekly saving against no pumped storage and the marginal saving per extra MWh of basin. Check one size against PyPSA.

In [ ]:
def pumped_value(sizes) -> pd.DataFrame:
    """Weekly saving (EUR) and marginal saving per added MWh (EUR/MWh) for each upper-basin size."""
    ...  # YOUR CODE HERE

In [ ]:
value = pumped_value([600, 1200, 2400, 4800, 9600])
ref = pypsa_week(pumped={"p_nom": 300, "e_nom": 2400, "eta": 0.866})
assert np.isclose(reservoir_week()["cost"] - value.loc[2400, "saving"], ref.objective, rtol=1e-9)
assert value.saving.is_monotonic_increasing
marginal = value.marginal_saving_per_MWh.dropna()
assert marginal.iloc[-1] < marginal.iloc[0], "the last MWh of basin is worth less than the first"
value.round(1)

**7. Interpretation**

What limits the value of a bigger upper basin in this week? Compare the marginal saving per MWh of basin with what a MWh of storage capacity might cost to build, and say what a planner would need to look at beyond one week.

*Your answer here.*